In [1]:
import pandas as pd
import pyodbc 
import numpy as np
from typing import List, Any

conn_str = (
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=localhost\\SQLEXPRESS;'
    'DATABASE=Company1;'
    'Trusted_Connection=yes;'
)


In [2]:
def get_db_connection():
    connection = pyodbc.connect(conn_str)
    return connection

In [19]:
sql_insert_customer = """
            INSERT INTO dbo.Customers (customer_id, name, gender, birthdate, phone, email, city)
            VALUES (?, ?, ?, ?, ?, ?, ?)
"""


sql_insert_branch = """
            INSERT INTO dbo.Branches (branch_id, branch_name, city, address)
            VALUES (?, ?, ?, ?)
"""

sql_insert_product = """
            INSERT INTO dbo.Products (product_id, product_name, category, price, supplier)
            VALUES (?, ?, ?, ?, ?)
"""

sql_insert_sale = """
            INSERT INTO dbo.Sales (sale_id, customer_id, branch_id, saledate, total_amount)
            VALUES (?, ?, ?, ?, ?)
"""

sql_insert_saledetails = """
            INSERT INTO dbo.SaleDetails (sale_details_id, sale_id, product_id, quantity, unitprice, totalprice)
            VALUES (?, ?, ?, ?, ?, ?)
"""

In [4]:
def add_data(data, sql_insert):
    try:
        # 1. open DB connection
        conn = get_db_connection()
        # 2. Create cursor to run command(s) on DB
        cursor = conn.cursor()
       #''' row_data = [] 
        #for key in data:
         #   row_data.append(data[key])'''

        # 4. Execute the SQL command with the data from "customer" dictionary
        cursor.execute(
            sql_insert,
            data
        )
        # 5. Save the changes (phisically) in the database
        conn.commit()

        # 6. Return success message
        return True, "Data added successfully"

    except Exception as e:
        return False, str(e)

    finally:
        # 6. Always close the connection
        conn.close()

In [3]:
def convert_df_to_native_types(df: pd.DataFrame) -> List[List[Any]]:
    """
    Convert DataFrame with numpy types to list of lists with native Python types.
    """
    data = []
    
    # Iterate through each row
    for row in df.itertuples(index=False, name=None):
        converted_row = []
        for value in row:
            # Convert numpy.int64 to Python int
            if isinstance(value, np.integer):
                converted_row.append(int(value))
            # Convert numpy.float64 to Python float
            elif isinstance(value, np.floating):
                converted_row.append(float(value))
            # Convert numpy.bool_ to Python bool
            elif isinstance(value, np.bool_):
                converted_row.append(bool(value))
            # Convert pandas.Timestamp to Python datetime
            elif isinstance(value, pd.Timestamp):
                converted_row.append(value.to_pydatetime())
            # Convert NaN/None to None
            elif pd.isna(value):
                converted_row.append(None)
            # Keep as-is for native types
            else:
                converted_row.append(value)
        data.append(converted_row)
    
    return data


In [13]:
customers = pd.read_csv('data/Customers.csv')
branches = pd.read_csv('data/Branches.csv')
products = pd.read_csv('data/Products.csv')
sales = pd.read_csv('data/Sales_50000.csv')
sales_details = pd.read_csv('data/SaleDetails_200000.csv')

customers_native_data = convert_df_to_native_types(customers)
branches_native_data = convert_df_to_native_types(branches)
products_native_data = convert_df_to_native_types(products)
sales_native_data = convert_df_to_native_types(sales)
sales_details_native_data = convert_df_to_native_types(sales_details)

In [ ]:
# one of the powerful way to convert  DataFrame to list of lists with native Python types
df_converted = customers.astype(object)
data_list = df_converted.values.tolist()

print(list(customers.loc[0]))

print(data_list[0])

print(customers_native_data[0])

[np.int64(1), 'Mostafa Youssef', 'Male', '10/22/1976', np.int64(1057664844), 'mostafa.youssef6067@yahoo.com', 'Alexandria']
[1, 'Mostafa Youssef', 'Male', '10/22/1976', 1057664844, 'mostafa.youssef6067@yahoo.com', 'Alexandria']
[1, 'Mostafa Youssef', 'Male', '10/22/1976', 1057664844, 'mostafa.youssef6067@yahoo.com', 'Alexandria']


In [10]:
for row_data in customers_native_data:
    add_data(row_data, sql_insert_customer)

In [14]:
for row_data in branches_native_data:
    add_data(row_data, sql_insert_branch)

In [16]:
for row_data in products_native_data:
    add_data(row_data, sql_insert_product)

In [17]:
for row_data in sales_native_data:
    add_data(row_data, sql_insert_sale)

In [20]:
for row_data in sales_details_native_data:
    add_data(row_data, sql_insert_saledetails)

In [4]:
sql_select = """
            SELECT
                c.customer_id,
                c.name AS customer_name,
                c.gender,
                c.birthdate,
                c.city AS customer_city,
                b.branch_id,
                b.branch_name,
                b.city AS branch_city,
                b.address AS branch_address,
                s.sale_id,
                s.saledate,
                s.total_amount,
                p.product_id,
                p.product_name,
                p.category,
                p.price,
                sd.quantity
            FROM Sales s
            JOIN Customers c ON s.customer_id = c.customer_id
            JOIN Branches b ON s.branch_id = b.branch_id
            JOIN SaleDetails sd ON s.sale_id = sd.sale_id
            JOIN Products p ON sd.product_id = p.product_id
        """


In [15]:
def get_all_orders(select):
    try:
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute(select)
        rows = cursor.fetchall()
        orders = []
        for row in rows:
            order = {
                "customer": {
                    "id": row.customer_id,
                    "name": row.customer_name,
                    "gender": row.gender,
                    "birthdate": row.birthdate.strftime("%Y-%m-%d") if row.birthdate else None,
                    "city": row.customer_city
                },
                "branch": {
                    "id": row.branch_id,
                    "name": row.branch_name,
                    "city": row.branch_city,
                    "address": row.branch_address
                },
                "sale": {
                    "id": row.sale_id,
                    "date": row.saledate.strftime("%Y-%m-%d") if row.saledate else None,
                    "total_amount": float(row.total_amount)  # 👈 هنا حول Decimal لـ float
                },
                   "product": {
                    "id": row.product_id,
                    "name": row.product_name,
                    "category": row.category,
                    "price": float(row.price),
                    "quantity": row.quantity
                }
            }

            orders.append(order)

        return True, orders

    except Exception as e:
        return False, str(e)
    finally:
        conn.close() 

In [16]:
success, orders = get_all_orders(sql_select)

In [18]:
print(success)
print(orders[0])

True
{'customer': {'id': 6495, 'name': 'Dina Ibrahim', 'gender': 'Female', 'birthdate': '1999-02-28', 'city': 'Giza'}, 'branch': {'id': 153, 'name': 'Abu Rawash Branch', 'city': 'Giza', 'address': '13 Industrial Rd, Abu Rawash, Giza'}, 'sale': {'id': 41906, 'date': '2022-01-07', 'total_amount': 9578.21}, 'product': {'id': 7297, 'name': 'Stationery Product 7297', 'category': 'Stationery', 'price': 224.38, 'quantity': 1}}


In [19]:
import pymongo
from pymongo import MongoClient

In [21]:
client = MongoClient('mongodb://localhost:27017/')

db = client["Retali_Business"]

orders_col = db["Orders"]

In [22]:
def insert_many_orders(orders_list):
    """
    orders_list: list of dictionaries, each containing full order info from SQL
    """
    try:
        # Attempt to insert multiple order documents
        result = orders_col.insert_many(orders_list)

        # Check if documents were actually inserted
        if result.inserted_ids:
            return True, f"{len(result.inserted_ids)} orders inserted successfully!"
        else:
            return False, "No orders were inserted"

    except Exception as e:
        # Handle any error during insert operation
        return False, str(e)


In [23]:
if success:
    success_insert, message = insert_many_orders(orders)
    if success_insert:
        print(message)
    else:
        print("Error inserting orders:", message)
else:
    print("Error fetching orders:", orders)


200000 orders inserted successfully!
